# **Multi-Model Reliability Evaluation (IndoPref)**

> Notebook ini digunakan untuk mengevaluasi reliabilitas dan bias posisi lima model `16-bit BF16` pada dataset IndoPref menggunakan paket `llm_exp`.

**Penulis:** [Rozhak](https://github.com/RozhakDev) &nbsp;|&nbsp; **Repositori:** [RozhakDev/LLMReliability](https://github.com/RozhakDev/LLMReliability) &nbsp;|&nbsp; **Dataset:** IndoPref

## **1. Setup Environment**

Bagian ini menyiapkan runtime Google Colab, seluruh dependensi proyek, dan pengaturan workspace sebelum evaluasi dijalankan.

### **1.1 Instalasi Dependensi**

Google Drive dihubungkan ke runtime Colab terlebih dahulu agar dependensi proyek dapat diinstal langsung dari berkas `requirements.txt`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!pip install -q -r /content/drive/MyDrive/LLMReliabilityEXP/Experiments/requirements.txt

### **1.2 Impor Modul**

Ekstensi `autoreload` diaktifkan bersama pendaftaran direktori `src` dan `scripts` ke `sys.path`, sehingga seluruh modul `llm_exp` dapat diimpor dalam satu sel.

In [ ]:
import sys
from pathlib import Path
import os

from dotenv import load_dotenv
from IPython.display import display
import pandas as pd

EXPERIMENTS_DIR = Path(
    os.environ.get("EXPERIMENTS_DIR", "/content/drive/MyDrive/LLMReliabilityEXP/Experiments")
)
for sub_dir in ("src", "scripts"):
    if (entry := str(EXPERIMENTS_DIR / sub_dir)) not in sys.path:
        sys.path.insert(0, entry)

from llm_exp.config import bootstrap_environment, load_model_registry
from llm_exp.data.indopref_loader import prepare_evaluation_dataset
from llm_exp.utils import log_hardware_status, setup_logger
from run_comparison import execute_comparison
from run_model import execute_model_pipeline

### **1.3 Pengaturan Workspace**

Berkas `.env` pada direktori `Experiments` dimuat untuk menentukan lokasi cache Hugging Face di disk lokal Colab agar penyimpanan Google Drive tidak terbebani.

In [ ]:
if not (EXPERIMENTS_DIR / "src" / "llm_exp").is_dir():
    raise FileNotFoundError(f"Experiments directory not found: {EXPERIMENTS_DIR}")

ENV_FILE = (
    EXPERIMENTS_DIR / ".env"
    if (EXPERIMENTS_DIR / ".env").is_file()
    else EXPERIMENTS_DIR / ".env.example"
)
load_dotenv(ENV_FILE, override=False)

if cache_dir := os.environ.get("MODELS_CACHE_DIR", "").strip():
    os.environ["HF_HOME"] = cache_dir
    os.environ["HF_HUB_CACHE"] = os.path.join(cache_dir, "hub")
    os.environ["HF_DATASETS_CACHE"] = os.path.join(cache_dir, "datasets")

## **2. Pengaturan Eksperimen**

Parameter eksekusi ditetapkan di bagian ini bersama pemeriksaan kesiapan perangkat keras sebelum inferensi dimulai.

### **2.1 Parameter Eksekusi**

Sel ini menampung kontrol utama eksekusi, meliputi tahap pipeline (`STAGE`), batas sampel uji (`MAX_TASKS`), tampilan grafik inline, serta opsi hitung ulang agregasi.

In [ ]:
RECOMPUTE_COMPARISON = True
STAGE = "all"
MAX_TASKS = None
SHOW_INLINE = True

PIPELINE_OPTIONS = {
    "stage": STAGE,
    "env_file": str(ENV_FILE),
    "max_tasks": MAX_TASKS,
    "show_inline": SHOW_INLINE,
}

### **2.2 Verifikasi Hardware**

Ketersediaan GPU, kapasitas VRAM, dan sisa ruang disk di `/content/models` diperiksa sebelum daftar lima model aktif dari `configs/models.yaml` ditampilkan.

In [ ]:
boot_logger = setup_logger("llm_exp.colab")
cfg, paths = bootstrap_environment(env_file=ENV_FILE, logger=boot_logger)
log_hardware_status(
    boot_logger,
    cache_dir=paths.models_cache_dir,
    min_vram_gb=cfg.inference.min_vram_gb,
    min_disk_gb=cfg.inference.min_disk_gb,
)

registry = load_model_registry(cfg.models_config_path)
models_df = pd.DataFrame(
    [
        {
            "key": spec.key,
            "model": spec.display_name,
            "hf_id": spec.hf_id,
            "loader": spec.loader_type,
            "est_vram_gb": spec.estimated_vram_gb,
            "gated": spec.requires_gated_access,
        }
        for spec in registry.values()
    ]
)
display(models_df)

## **3. Dataset IndoPref**

Split evaluasi IndoPref disiapkan melalui loader proyek, kemudian ditampilkan contoh baris beserta distribusi sampel pada tiap domain.

In [ ]:
data_logger = setup_logger(
    "llm_exp.data",
    log_dir=paths.logs_dir,
    run_name="prepare_data",
    logging_config=cfg.logging,
)
df_eval = prepare_evaluation_dataset(paths=paths, logger=data_logger, config=cfg)

display(df_eval[["original_id", "domain"]].head(3))
display(df_eval["domain"].value_counts().to_frame(name="samples"))

## **4. Evaluasi per Model**

Setiap model dieksekusi pada sel terpisah agar progres tersimpan per baris di berkas JSONL dan dapat dilanjutkan otomatis apabila sesi Colab terputus. Log ditulis per sesi ke `logs/YYYY-MM-DD/`, sedangkan VRAM dan cache `/content/models` dibersihkan setelah seluruh tugas satu model selesai.

### **4.1 Qwen3 32B**

Qwen3 32B berperan sebagai model baseline utama dengan loader `causal_lm` pada mode non-thinking.

In [ ]:
execute_model_pipeline("qwen3_32b", **PIPELINE_OPTIONS)

### **4.2 GoTo-AI Gemma 3 27B**

GoTo-AI Gemma 3 27B merupakan model 27B dengan spesialisasi bahasa Indonesia yang dimuat melalui loader `gemma3`.

In [ ]:
execute_model_pipeline("gemma3_27b_goto", **PIPELINE_OPTIONS)

### **4.3 Aya Expanse 32B**

Aya Expanse 32B merupakan model multibahasa 32B dengan loader `causal_lm`. Repositori ini berstatus *gated*, sehingga token Hugging Face harus memiliki izin akses terlebih dahulu.

In [ ]:
execute_model_pipeline("aya_expanse_32b", **PIPELINE_OPTIONS)

### **4.4 Mistral Small 3.1 24B**

Mistral Small 3.1 24B difokuskan pada kemampuan instruction-following 24B dengan loader `mistral3`.

In [ ]:
execute_model_pipeline("mistral_small_3_1_24b", **PIPELINE_OPTIONS)

### **4.5 GLM-4 32B 0414**

GLM-4 32B 0414 berfungsi sebagai pembanding kelas 32B pada loader `causal_lm` yang memerlukan opsi `trust_remote_code`.

In [ ]:
execute_model_pipeline("glm_4_32b", **PIPELINE_OPTIONS)

## **5. Komparasi Hasil**

Metrik dari seluruh model yang telah selesai dievaluasi digabungkan pada bagian ini. Grafik perbandingan disimpan ke `results/comparison/`, lalu tabel ringkasan dan akurasi per domain ditampilkan di bawahnya.

In [ ]:
execute_comparison(
    env_file=str(ENV_FILE),
    recompute_from_jsonl=RECOMPUTE_COMPARISON,
    show_inline=SHOW_INLINE,
)

summary_csv = paths.comparison_dir / cfg.artifacts.comparison_summary_csv
domain_csv = paths.comparison_dir / cfg.artifacts.comparison_domain_accuracy_csv

display(pd.read_csv(summary_csv))
if domain_csv.is_file():
    display(pd.read_csv(domain_csv, index_col=0))